# Group 3 Challenge: Campus Coffee and Print Shop POS

Your team is building the point-of-sale billing system for a student-run campus cafe.
Fix the broken starter code across all three tasks so every assertion passes.

**Scoring:** Task 1 (`30 pts`) + Task 2 (`35 pts`) + Task 3 (`35 pts`) = **`100 pts`**.

In [ ]:
SCORES = {"task_1": 0, "task_2": 0, "task_3": 0}
print("Score tracker initialized. Complete Tasks 1 to 3 below!")

---
## Task 1: Encapsulation `[30 pts]` — Protect `OrderTicket` State

Fix the `OrderTicket` class below to enforce these rules:
1. `__init__(self, ticket_id: str, discount_rate: float = 0.0)`:
   - Raise `ValueError("discount_rate must be between 0.0 and 0.5")` if not `0.0 <= discount_rate <= 0.5`.
   - Store `self._ticket_id`, `self._discount_rate`, and `self._items = []`.
2. Read-only properties:
   - `ticket_id` -> returns `self._ticket_id`
   - `items` -> returns `tuple(self._items)` (caller cannot mutate internal list!)
3. Validated `discount_rate` property:
   - `@property` `discount_rate` returns `self._discount_rate`.
   - `@discount_rate.setter` raises `ValueError("discount_rate must be between 0.0 and 0.5")` if outside `[0.0, 0.5]`.
4. Method `add_item(self, name: str, price: float)`:
   - Raise `ValueError("price must be positive")` if `price <= 0`.
   - Append `(name, float(price))` to `self._items`.

In [ ]:
class OrderTicket:
    def __init__(self, ticket_id: str, discount_rate: float = 0.0):
        # BUG 1: Missing discount_rate validation [0.0, 0.5]
        # BUG 2: Public attributes instead of protected (_ticket_id, _discount_rate, _items)
        self.ticket_id = ticket_id
        self.discount_rate = discount_rate
        self.items = []

    # TODO: Implement read-only properties: ticket_id, items (returning a tuple)
    # TODO: Implement discount_rate @property and @discount_rate.setter with validation [0.0, 0.5]

    def add_item(self, name: str, price: float):
        # BUG 3: Missing positive price validation
        self.items.append((name, float(price)))

    def subtotal(self) -> float:
        return sum(price for _, price in self._items)

    def final_total(self) -> float:
        return round(self.subtotal() * (1 - self._discount_rate), 2)

    def summary(self) -> str:
        return f"Ticket #{self._ticket_id} ({len(self._items)} items)"

In [ ]:
# RUN THIS CELL TO VERIFY TASK 1 (30 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    bad_init = False
    try:
        OrderTicket("T-0", 0.9)
    except ValueError:
        bad_init = True
    assert bad_init, "OrderTicket('T-0', 0.9) must raise ValueError."

    t = OrderTicket("A-101", 0.1)
    assert hasattr(t, "_ticket_id") and hasattr(t, "_discount_rate") and hasattr(t, "_items"), "Use protected attributes (_ticket_id, _discount_rate, _items)."
    assert t.ticket_id == "A-101" and t.discount_rate == 0.1, "Properties ticket_id and discount_rate must return initial values."

    ro = False
    try:
        t.ticket_id = "B-202"
    except AttributeError:
        ro = True
    assert ro, "Property 'ticket_id' must be read-only."

    assert isinstance(t.items, tuple), "Property 'items' must return a tuple."

    bad_set = False
    try:
        t.discount_rate = -0.1
    except ValueError:
        bad_set = True
    assert bad_set, "Setting discount_rate < 0.0 must raise ValueError."

    bad_price = False
    try:
        t.add_item("Water", 0)
    except ValueError:
        bad_price = True
    assert bad_price, "add_item('Water', 0) must raise ValueError."

    t.add_item("Latte", 100.0)
    assert t.items == (("Latte", 100.0),) and t.final_total() == 90.0, "Check items tuple and final_total()."

    SCORES["task_1"] = 30
    print("[PASS] Task 1 (30 / 30 pts): OrderTicket encapsulation verified.")
except Exception as e:
    SCORES["task_1"] = 0
    print(f"[FAIL] Task 1 (0 / 30 pts): {e}")

---
## Task 2: Inheritance `[35 pts]` — `StudentOrder` and `BulkOrder`

Fix `StudentOrder` and `BulkOrder` so they inherit from `OrderTicket`:
1. `StudentOrder(OrderTicket)`:
   - `__init__(self, ticket_id: str, discount_rate: float, campus_id: str)`: call `super().__init__(ticket_id, discount_rate)` and store `self.campus_id = campus_id`.
   - Override `final_total(self) -> float`: subtract an extra flat `10.0` from `super().final_total()`, clamped at `0.0`: `round(max(0.0, super().final_total() - 10.0), 2)`.
   - Extend `summary(self) -> str`: return `f"{super().summary()} | Student {self.campus_id}"`.
2. `BulkOrder(OrderTicket)`:
   - `__init__(self, ticket_id: str, discount_rate: float, service_fee: float)`: call `super().__init__(ticket_id, discount_rate)` and store `self.service_fee = service_fee`.
   - Override `final_total(self) -> float`: add `self.service_fee` to `super().final_total()`: `round(super().final_total() + self.service_fee, 2)`.
   - Extend `summary(self) -> str`: return `f"{super().summary()} | Bulk (+{self.service_fee:.2f} fee)"`.

In [ ]:
class StudentOrder(OrderTicket):
    def __init__(self, ticket_id: str, discount_rate: float, campus_id: str):
        # BUG 1: Missing super().__init__(ticket_id, discount_rate)
        self.campus_id = campus_id

    def final_total(self) -> float:
        # BUG 2: Subtract 10.0 from super().final_total(), clamped at 0.0
        return 0.0

    def summary(self) -> str:
        # BUG 3: Extend super().summary()
        return self.campus_id


class BulkOrder(OrderTicket):
    def __init__(self, ticket_id: str, discount_rate: float, service_fee: float):
        # BUG 1: Missing super().__init__(ticket_id, discount_rate)
        self.service_fee = service_fee

    def final_total(self) -> float:
        # BUG 2: Add self.service_fee to super().final_total()
        return 0.0

    def summary(self) -> str:
        # BUG 3: Extend super().summary()
        return f"Bulk {self.service_fee}"

In [ ]:
# RUN THIS CELL TO VERIFY TASK 2 (35 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    assert issubclass(StudentOrder, OrderTicket) and issubclass(BulkOrder, OrderTicket), "Both subclasses must inherit from OrderTicket."

    so = StudentOrder("S-1", 0.2, "2024-001")
    so.add_item("Americano", 100.0)
    assert so.final_total() == 70.0, f"StudentOrder expected 70.0 (100*0.8 - 10), got {so.final_total()}."
    assert so.summary() == "Ticket #S-1 (1 items) | Student 2024-001", f"Unexpected summary: {so.summary()}"

    bo = BulkOrder("B-1", 0.1, 25.0)
    bo.add_item("Print Pack", 200.0)
    assert bo.final_total() == 205.0, f"BulkOrder expected 205.0 (200*0.9 + 25), got {bo.final_total()}."
    assert bo.summary() == "Ticket #B-1 (1 items) | Bulk (+25.00 fee)", f"Unexpected summary: {bo.summary()}"

    SCORES["task_2"] = 35
    print("[PASS] Task 2 (35 / 35 pts): Inheritance and super() verified.")
except Exception as e:
    SCORES["task_2"] = 0
    print(f"[FAIL] Task 2 (0 / 35 pts): {e}")

---
## Task 3: Polymorphism and Duck Typing `[35 pts]` — `PromoVoucher` & `calculate_daily_revenue`

Fix the code below to achieve clean polymorphism:
1. `PromoVoucher` must **NOT** inherit from `OrderTicket` (standalone duck-typed class).
   - Rename/implement `final_total(self) -> float` returning `round(float(self.fixed_amount), 2)`.
2. Refactor `calculate_daily_revenue(tickets: list) -> float`:
   - Remove **all** `isinstance(...)` checks!
   - Loop through `tickets`, sum `t.final_total()`, and return `round(total, 2)`.

In [ ]:
class PromoVoucher:
    def __init__(self, code: str, fixed_amount: float):
        self.code = code
        self.fixed_amount = fixed_amount

    # BUG 1: Wrong method name for duck typing! Should be final_total(self) -> float
    def redeem_value(self) -> float:
        return round(float(self.fixed_amount), 2)


def calculate_daily_revenue(tickets: list) -> float:
    # BUG 2: Fragile isinstance() branching misses PromoVoucher!
    # Refactor into a clean polymorphic loop with ZERO isinstance() calls.
    total = 0.0
    for t in tickets:
        if isinstance(t, StudentOrder):
            total += t.final_total()
        elif isinstance(t, BulkOrder):
            total += t.final_total()
    return round(total, 2)

In [ ]:
# RUN THIS CELL TO VERIFY TASK 3 (35 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    assert not issubclass(PromoVoucher, OrderTicket), "PromoVoucher must NOT inherit from OrderTicket."
    assert hasattr(PromoVoucher, "final_total"), "PromoVoucher must implement final_total()."

    assert "isinstance" not in calculate_daily_revenue.__code__.co_names, "Remove all isinstance() calls from calculate_daily_revenue()."

    s = StudentOrder("S2", 0.0, "2024-002")
    s.add_item("Tea", 60.0)  # 50.0
    b = BulkOrder("B2", 0.0, 15.0)
    b.add_item("Binding", 100.0)  # 115.0
    v = PromoVoucher("EVENT-50", 50.0)  # 50.0

    rev = calculate_daily_revenue([s, b, v])
    assert rev == 215.0, f"Expected total revenue 215.0, got {rev}."

    SCORES["task_3"] = 35
    print("[PASS] Task 3 (35 / 35 pts): Polymorphism and duck typing verified.")
except Exception as e:
    SCORES["task_3"] = 0
    print(f"[FAIL] Task 3 (0 / 35 pts): {e}")

---
## Final Scoreboard

In [ ]:
# RUN THIS CELL TO VIEW YOUR FINAL COMPETITION SCORE
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
total = SCORES["task_1"] + SCORES["task_2"] + SCORES["task_3"]
print("=" * 42)
print("DS212 GROUP COMPETITION SCOREBOARD")
print("=" * 42)
print(f"Task 1 (Encapsulation): {SCORES['task_1']:>2} / 30 pts")
print(f"Task 2 (Inheritance):   {SCORES['task_2']:>2} / 35 pts")
print(f"Task 3 (Polymorphism):  {SCORES['task_3']:>2} / 35 pts")
print("-" * 42)
print(f"TOTAL SCORE:           {total:>3} / 100 pts")
if total == 100:
    print("STATUS: ALL CHALLENGES COMPLETED [100%]")
else:
    print("STATUS: IN PROGRESS — KEEP DEBUGGING!")
print("=" * 42)